# 01: DNA Language Model Tokenization Comparison

Tokenization is how a language model chunks its input into discrete units before processing. Modern text-based language models like GPT tokenize English into subword pieces via byte-pair encoding (BPE). DNA language models tokenize nucleotide sequences into units that vary by design choice:

- **Fixed k-mer tokenization**: sequences are sliced into equal-length chunks (e.g., 6 nucleotides per token).
- **Byte-pair encoding (BPE)**: variable-length tokens are learned from training data, with common substrings assigned single tokens.
- **Single-nucleotide tokenization**: each base becomes its own token.

This notebook applies all three tokenization strategies to the same 200 bp DNA sequence and displays what each produces. Three foundation models are compared:

| Model | Tokenization strategy | Approximate vocabulary size |
|---|---|---|
| Nucleotide Transformer v2 (500M) | Fixed 6-mer | ~4,105 |
| DNABERT-2 (117M) | Byte-pair encoding (BPE) | ~4,096 |
| HyenaDNA (medium) | Single nucleotide | ~12 |

The comparison is illustrative rather than benchmarking. Actual performance is measured in the next notebook (`02_perplexity_benchmark.ipynb`), which evaluates masked language modeling loss on real *Anopheles gambiae* sequences.

**Kernel:** Python 3.11 (Mathieson-LM)

## Setup

Standard project setup: path resolution, package imports, and HuggingFace cache configuration. HuggingFace's `AutoTokenizer` class provides a uniform API for loading tokenizers across model families, so the same code loads all three models below.

Model files are cached under `models/` in this project directory rather than the default global cache. This keeps model downloads self-contained per project and easy to relocate.

In [ ]:
# Standard library imports for filesystem operations
import sys, os, warnings
from pathlib import Path

# Suppress noisy deprecation warnings from third-party libraries
warnings.filterwarnings('ignore')

# --- Project path resolution ---
# NOTEBOOK_DIR is where this notebook lives (Protein-LM/notebooks/)
# PROJECT_ROOT is one level up (Protein-LM/)
# MODELS_DIR is where HuggingFace will cache downloaded model files
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

# --- HuggingFace cache configuration ---
# Setting HF_HOME redirects all HuggingFace downloads (models, tokenizers, datasets)
# to the specified directory instead of the default (~/.cache/huggingface).
# This must be set BEFORE importing transformers for it to take effect.
os.environ['HF_HOME'] = str(MODELS_DIR)

# --- Scientific Python imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- HuggingFace transformers import ---
# AutoTokenizer selects the right tokenizer class automatically based on the model ID
from transformers import AutoTokenizer

print(f'HF model cache: {MODELS_DIR}')
print('Ready.')

## Test sequence

A 200 base pair synthetic DNA sequence containing several biologically recognized motifs:

- **TATA box** (`TATAAA`): a canonical eukaryotic promoter element found upstream of many transcription start sites. Appears twice in the sequence below.
- **CAAT box** (`GGCCAATCT`): a second common promoter element.
- **Poly-A stretch** (`AAAA...`): often filtered in real analyses due to its unusual base composition and repetitive structure.
- **GC-rich region** (`CGCGCG...`): a pattern commonly found near active promoters (CpG islands).

The remaining sequence is unstructured filler to give the tokenizers something less predictable to work with. The sequence is synthetic rather than mosquito-derived because the goal here is only to visualize how each tokenizer partitions a known input. The same tools are applied to real *An. gambiae* sequences in the following notebook.

In [ ]:
# --- Define the test sequence as a concatenation of labeled subsequences ---
# Comments to the right indicate what each block represents
TEST_SEQ = (
    "ACGTACGTACGTACGTACGT"          # 20 bp filler (repetitive but not motif-like)
    "TATAAA"                          # First TATA box
    "CCTGCAGGATCGATCGATCG"           # 20 bp filler
    "GGCCAATCT"                       # CAAT box
    "ATCGATCGATCGATCGATCG"           # 20 bp filler
    "AAAAAAAAAAAAAAAAAAAA"           # 20 bp poly-A stretch
    "CGTAGCTACGTAGCTAGCTA"           # 20 bp filler
    "TATAAA"                          # Second TATA box (repeated motif)
    "GCTAGCTAGCTAGCTAGCTA"           # 20 bp filler
    "CGCGCGCGCGCGCGCGCGCG"           # 20 bp GC-rich pattern
    "TACG"                            # 4 bp trailing
)

# --- Basic sequence statistics ---
# GC content is a common summary of nucleotide composition; A. gambiae is ~44%
gc_count = TEST_SEQ.count("G") + TEST_SEQ.count("C")
gc_content = gc_count / len(TEST_SEQ)

print(f'Test sequence length: {len(TEST_SEQ)} bp')
print(f'GC content: {gc_content:.1%}')
print(f'\nSequence (60 bp per line):')

# Print the sequence in blocks of 60 bp with position markers
for i in range(0, len(TEST_SEQ), 60):
    print(f'  {i:3d}  {TEST_SEQ[i:i+60]}')

## Loading the tokenizers

Only the tokenizers are loaded here, not the full transformer weights. Tokenizers are lightweight artifacts (typically a vocabulary file plus tokenization rules totaling a few hundred kilobytes) and do not require GPU. The full model weights are loaded in the following notebook.

On first execution, tokenizer files are downloaded from HuggingFace's model registry and cached in `models/hub/` (per the `HF_HOME` environment variable set above). Subsequent runs load from that local cache.

The `trust_remote_code=True` argument permits loading tokenizer code from the model repositories. This is required for models like DNABERT-2 and HyenaDNA that define custom tokenizer classes not built into the transformers library.

In [ ]:
# --- Model identifiers on HuggingFace ---
# Each entry maps a human-readable label to the HuggingFace model ID
# (of the form username/model-name)
MODELS = {
    'NT-v2 (6-mer)':          'InstaDeepAI/nucleotide-transformer-v2-500m-multi-species',
    'DNABERT-2 (BPE)':        'zhihan1996/DNABERT-2-117M',
    'HyenaDNA (single-nt)':   'LongSafari/hyenadna-medium-450k-seqlen-hf',
}

# --- Load each tokenizer into a dictionary keyed by label ---
# Wrapped in try/except so one model failing does not abort the others
tokenizers = {}
for name, model_id in MODELS.items():
    print(f'Loading {name:<24} from {model_id}...')
    try:
        # AutoTokenizer.from_pretrained downloads (or loads from cache) the tokenizer
        # trust_remote_code=True allows custom tokenizer classes defined in the model repo
        tokenizers[name] = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        vocab_size = tokenizers[name].vocab_size
        print(f'  OK  vocabulary size: {vocab_size:,}')
    except Exception as e:
        # Report the failure but continue so partial success is still informative
        print(f'  FAIL: {e.__class__.__name__}: {str(e)[:100]}')
        tokenizers[name] = None

# --- Summary of successful loads ---
loaded_count = sum(t is not None for t in tokenizers.values())
print(f'\n{loaded_count}/{len(MODELS)} tokenizers loaded')

## Tokenizing the same sequence three ways

Each tokenizer converts the same 200 bp input into a sequence of token IDs. The number of tokens produced reflects the tokenization strategy:

- Fixed 6-mer tokenizers produce approximately `sequence_length / 6` tokens (plus a few special tokens for sequence boundaries).
- BPE produces a variable count depending on which learned vocabulary entries match the input.
- Single-nucleotide tokenization produces roughly one token per base.

Expected token counts for this 200 bp input:

| Tokenizer | Approximate token count |
|---|---|
| NT-v2 (6-mer) | ~35 (200 / 6 + special tokens) |
| DNABERT-2 (BPE) | Variable, typically 40 to 80 |
| HyenaDNA (single-nt) | ~202 (200 + special tokens) |

In [ ]:
def analyze_tokenization(name, tokenizer, sequence):
    """
    Tokenize a sequence and return a dictionary summary.

    Parameters
    ----------
    name : str
        Human-readable label for the model.
    tokenizer : PreTrainedTokenizer
        A loaded HuggingFace tokenizer.
    sequence : str
        The DNA sequence to tokenize.

    Returns
    -------
    dict with keys: name, num_tokens, ids, tokens, vocab_size
    Returns None if the tokenizer is None (failed to load).
    """
    if tokenizer is None:
        return None

    # tokenizer.encode() converts the string into a list of integer token IDs
    # add_special_tokens=True prepends [CLS] and appends [SEP] tokens (BERT-style boundaries)
    ids = tokenizer.encode(sequence, add_special_tokens=True)

    # convert_ids_to_tokens maps integer IDs back to their string forms for inspection
    tokens = tokenizer.convert_ids_to_tokens(ids)

    return {
        'name': name,
        'num_tokens': len(ids),
        'ids': ids,
        'tokens': tokens,
        'vocab_size': tokenizer.vocab_size,
    }

# --- Run analyze_tokenization for each loaded tokenizer ---
results = {}
for name, tok in tokenizers.items():
    r = analyze_tokenization(name, tok, TEST_SEQ)
    if r is not None:
        results[name] = r

# --- Print a summary table ---
# The bases_per_token column shows the compression ratio
# (higher = fewer tokens per base of input = more compressed representation)
print(f'{"Model":<25} {"Vocab":>10} {"Tokens":>10} {"Bases/token":>15}')
print('-' * 65)
for r in results.values():
    # Subtract 2 from num_tokens as a rough correction for special tokens ([CLS] and [SEP])
    bases_per_token = len(TEST_SEQ) / max(r['num_tokens'] - 2, 1)
    print(f"{r['name']:<25} {r['vocab_size']:>10,} {r['num_tokens']:>10,} {bases_per_token:>15.2f}")

## Actual token strings

Printing the first several tokens produced by each tokenizer. This shows the concrete vocabulary units each model treats as its atomic input pieces.

Notable patterns:

- Fixed k-mer tokens are always exactly the same length (6 bp for NT-v2).
- BPE tokens have variable length, reflecting learned frequency of substrings in the training corpus.
- Single-nucleotide tokens are always one character long.

The special tokens visible at the start of each list (`[CLS]`, `<cls>`, or similar) mark the beginning of the sequence and are part of every tokenized input in BERT-style models.

In [ ]:
# --- Print the first 15 tokens from each model side by side ---
# Each row shows: position index, the token as a string, its integer ID
for name, r in results.items():
    print(f'\n=== {name} ===')
    print(f'Total tokens: {r["num_tokens"]}')
    print(f'First 15 tokens (as strings):')

    # Iterate through the first 15 tokens (or fewer if the sequence is shorter)
    for i, tok in enumerate(r['tokens'][:15]):
        # repr() shows the string with quotes so whitespace/special chars are visible
        print(f'  {i:3d}  {repr(tok):<20}  id={r["ids"][i]}')

    # If more tokens exist, note how many were truncated from display
    if r['num_tokens'] > 15:
        print(f'  ...  ({r["num_tokens"] - 15} more tokens)')

## Motif detection: does the tokenizer treat biological motifs as single tokens?

A hypothesis specific to BPE tokenization is that biologically important sequence motifs, if they appear frequently enough in pretraining data, get compressed into single vocabulary entries. This is directly analogous to how English BPE tokenizers assign single tokens to common words like `the` or `and`.

This cell scans each model's tokenized output for common regulatory motifs (TATA box, CAAT box, GC-rich stretches, poly-A). A "hit" means the motif appears as a substring within one or more tokens. Interpreting the results:

- If a motif appears as its own token (or within a longer token), the tokenizer captures that biological pattern at the vocabulary level.
- If a motif is always split across multiple tokens, the tokenizer treats it as ordinary sequence content.
- Fixed k-mer tokenizers almost always split motifs unless they happen to align with the k-mer window boundaries.
- Single-nucleotide tokenizers never capture multi-base motifs since every token is one base.

In [ ]:
# --- Motifs to search for in the tokenized output ---
# TATAAA and CAAT are canonical eukaryotic promoter elements
# CGCG is characteristic of CpG islands (promoter regions)
# AAAA represents low-complexity poly-A stretches
MOTIFS_TO_HUNT = ['TATAAA', 'TATA', 'CAAT', 'CGCG', 'AAAA']

# --- For each model, check whether any tokens contain each motif ---
for name, r in results.items():
    print(f'\n=== {name} ===')

    for motif in MOTIFS_TO_HUNT:
        # Search all tokens for the motif as a substring (case-insensitive)
        # tok.upper() normalizes token casing since some tokenizers use lowercase strings
        hits = [tok for tok in r['tokens'] if motif in tok.upper()]

        if hits:
            # Deduplicate the list of matching tokens while preserving order
            # dict.fromkeys preserves insertion order in Python 3.7+
            uniq = list(dict.fromkeys(hits))
            # Show only the first 5 unique matches to keep output readable
            print(f'  {motif:<8} found in {len(hits)} token(s): {uniq[:5]}')
        else:
            print(f'  {motif:<8} not found as a substring in any token')

## Visualizing token boundaries

Rendering the tokenization by wrapping each token in brackets on the original sequence. This makes token boundaries visible at a glance and highlights the structural differences between strategies:

- Fixed 6-mer produces uniform 6-character blocks throughout.
- BPE produces irregular block sizes reflecting the learned vocabulary.
- Single-nucleotide produces one bracket per character.

Some tokenizers include leading special characters in token strings (`##` for BERT-style continuation, `▁` for SentencePiece word boundaries, `Ġ` for GPT-style whitespace prefixes). These are stripped for display so the underlying nucleotide content is visible.

In [ ]:
def render_tokenization(name, tokenizer, sequence, max_show=200):
    """
    Print a bracketed rendering of a sequence's tokenization.

    Each token appears wrapped in [square brackets]. This makes token
    boundaries visually apparent when reading the sequence linearly.

    Parameters
    ----------
    name : str
        Model label for the header.
    tokenizer : PreTrainedTokenizer or None
        A loaded tokenizer (skipped if None).
    sequence : str
        The DNA sequence to tokenize.
    max_show : int
        Approximate maximum number of characters to render before truncating.
    """
    if tokenizer is None:
        print(f'{name}: tokenizer not loaded, skipping')
        return

    # Encode WITHOUT special tokens so the bracketed output aligns with raw input positions
    ids = tokenizer.encode(sequence, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(ids)

    print(f'\n=== {name} ===')

    # --- Build the bracketed representation as a single string ---
    line = ''
    for tok in tokens:
        # Strip tokenizer-specific prefixes that would clutter the display:
        #   '##' is BERT-style subword continuation marker
        #   '\u2581' (Unicode SentencePiece marker) indicates word boundaries
        #   'Ġ' is used by GPT-style tokenizers to mark preceding whitespace
        clean = tok.replace('##', '').replace('\u2581', '').replace('\u0120', '')
        line += f'[{clean}]'

    # --- Wrap the long output line for readable printing ---
    # 90 characters per line is a comfortable width for most terminals
    for i in range(0, min(len(line), max_show * 3), 90):
        print(f'  {line[i:i+90]}')

    # If truncation happened, indicate that more output was omitted
    if len(line) > max_show * 3:
        print(f'  ... (truncated at {max_show} bp of source sequence)')

# --- Render each tokenizer's output on the test sequence ---
for name, tok in tokenizers.items():
    render_tokenization(name, tok, TEST_SEQ)

## Summary

The same 200 bp DNA sequence produces qualitatively different token representations depending on the tokenization strategy:

- **Nucleotide Transformer v2** emits fixed 6 bp blocks, giving predictable token positions but rigidly bound to the k-mer window alignment. A single-base insertion in the input would shift every downstream token.
- **DNABERT-2** emits variable-length BPE tokens; common motifs may map to single vocabulary entries, while unusual substrings get split into smaller pieces.
- **HyenaDNA** emits single-nucleotide tokens, preserving full base-pair resolution at the cost of longer token sequences downstream (relevant for models with limited context length).

Each strategy has tradeoffs for downstream modeling:

- Fixed k-mer is simplest and preserves codon structure (6 = 2 codons per token).
- BPE is more expressive but token boundaries depend on training data statistics.
- Single-nucleotide preserves maximum resolution but scales poorly with sequence length.

The next notebook (`02_perplexity_benchmark.ipynb`) quantifies how well each model, when applied to real mosquito DNA sequences, predicts the content of masked or held-out tokens. Lower prediction loss indicates a better fit between the model's pretrained knowledge and *An. gambiae* genomic content.